# Task
1. Load SMILES strings and parse with RDKit
2. Compute basic physicochemical descriptors
3. Compute Morgan/ECFP fingerprints

In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import Descriptors


In [2]:
df = pd.read_csv("~/smiles_prop/data/delaney-processed.csv")

print(df.head(5))

  Compound ID  ESOL predicted log solubility in mols per litre  \
0   Amigdalin                                           -0.974   
1    Fenfuram                                           -2.885   
2      citral                                           -2.579   
3      Picene                                           -6.618   
4   Thiophene                                           -2.232   

   Minimum Degree  Molecular Weight  Number of H-Bond Donors  Number of Rings  \
0               1           457.432                        7                3   
1               1           201.225                        1                2   
2               1           152.237                        0                0   
3               2           278.354                        0                5   
4               2            84.143                        0                1   

   Number of Rotatable Bonds  Polar Surface Area  \
0                          7              202.32   
1           

In [3]:
print(df.shape[0])
print(df.columns)

1128
Index(['Compound ID', 'ESOL predicted log solubility in mols per litre',
       'Minimum Degree', 'Molecular Weight', 'Number of H-Bond Donors',
       'Number of Rings', 'Number of Rotatable Bonds', 'Polar Surface Area',
       'measured log solubility in mols per litre', 'smiles'],
      dtype='object')


In [4]:
mol = Chem.MolFromSmiles("not a real smiles string")
print(mol)
print(type(mol))

x = None
print(type(x))
print(type(x) is None)
print(x is None)
print(mol is None)

None
<class 'NoneType'>
<class 'NoneType'>
False
True
True


[20:00:55] SMILES Parse Error: syntax error while parsing: not
[20:00:55] SMILES Parse Error: check for mistakes around position 3:
[20:00:55] not
[20:00:55] ~~^
[20:00:55] SMILES Parse Error: Failed parsing SMILES 'not' for input: 'not'


In [5]:

smiles_list = []

for row in range(df.shape[0]):
    mol = Chem.MolFromSmiles(df['smiles'][row])
    if mol is None:
        continue


    smiles_list.append({"smiles": df["smiles"][row],
                        "y": df["measured log solubility in mols per litre"][row],
                        "mol_weight": df["Molecular Weight"][row],
                        "logp": Descriptors.MolLogP(mol),
                        "tpsa": Descriptors.TPSA(mol),
                        "h_donors": Descriptors.NumHDonors(mol),
                        "h_acceptors": Descriptors.NumHAcceptors(mol)})

smiles_df = pd.DataFrame(smiles_list)

In [6]:
print(smiles_df.dtypes)
print("\n")
print(smiles_df.head(5))

smiles          object
y              float64
mol_weight     float64
logp           float64
tpsa           float64
h_donors         int64
h_acceptors      int64
dtype: object


                                              smiles     y  mol_weight  \
0  OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)... -0.77     457.432   
1                             Cc1occc1C(=O)Nc2ccccc2 -3.30     201.225   
2                               CC(C)=CCCC(C)=CC(=O) -2.06     152.237   
3                 c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43 -7.87     278.354   
4                                            c1ccsc1 -1.33      84.143   

      logp    tpsa  h_donors  h_acceptors  
0 -3.10802  202.32         7           12  
1  2.84032   42.24         1            2  
2  2.87800   17.07         0            1  
3  6.29940    0.00         0            0  
4  1.74810    0.00         0            1  


# QSAR baseline
1. split data: X_train, X_test, y_train, y_test
2. instantiate random forest
3. fir the model fit(X_train, y_train)
4. get predictions on data the model has never seen predict(X_test)
5. evaluate using rmse and r^2
6. inspect feature importance for the model

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


In [8]:

X = smiles_df[[#
    "mol_weight",
    "logp",
    "tpsa",
    "h_donors",
    "h_acceptors"
]]

y = smiles_df["y"]

X_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
model = RandomForestRegressor(n_estimators=400, # number of trees
                              random_state=42,
                              n_jobs=-1) # -1 to use all processors without jobs in parallel

In [10]:
model.fit(X_train, y_train) # trained the model

y_pred = model.predict(x_test)

results = pd.DataFrame({
    "Predicted": y_pred,
    "True": y_test.to_numpy()
})

print(results.head(5))

   Predicted   True
0  -2.629980 -2.540
1  -2.740558 -2.253
2  -2.164269 -2.484
3  -2.629980 -2.540
4  -6.855162 -7.200


In [11]:
rmse = mean_squared_error(y_pred=y_pred, y_true=y_test)**0.5
r2 = r2_score(y_pred=y_pred, y_true=y_test)

print(f"RMSE: {rmse}\nR^2: {r2}")

RMSE: 0.8439397712412684
R^2: 0.8493200872287214


# Scafold split implementation for better train/test split

In [12]:
from rdkit.Chem.Scaffolds import MurckoScaffold

scaffold_groups = set()

for smiles in smiles_df["smiles"]:
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        continue

    scaffold_mol = MurckoScaffold.GetScaffoldForMol(mol)
    scaffold_smiles = Chem.MolToSmiles(scaffold_mol)

    scaffold_groups.add(scaffold_smiles)


In [13]:
print(scaffold_groups)

{'', 'O=C1C=C(C2CCC(c3ccccc3)CC2)C(=O)c2ccccc21', 'S=C1Nc2ccccc2Nc2ncccc21', 'O=C1NC(=O)C(=O)C(=O)N1', 'C1CC2CCC3C4CC5SC5CC4CCC3C2C1', 'c1ccc2c(c1)ccc1c2ccc2c3ccccc3ccc21', 'O=C1Nc2cccnc2N(C2CC2)c2ncccc21', 'O=c1[nH]cnc2[nH]cnc12', 'O=C(Nc1ccccc1)c1ccccc1', 'O=C1CC(=O)N2c3ccccc3N=CN12', 'c1ccc(-c2cnc3ncncc3n2)cc1', 'c1cc2c3c(c4ccc5ccccc5c4cc3c1)CC2', 'c1ccsc1', 'O=C1NCCN1c1nccs1', 'O=C1CC(CCC2CCCCC2=O)CC(=O)N1', 'O=C1C=C2CCC3C4CCC(=O)C4CC(=O)C3C2CC1', 'O=c1ccc2ccccc2o1', 'O=C1NC(=O)C2(CCCCCC2)C(=O)N1', 'c1ccc2ccccc2c1', 'O=C(Nc1ccccc1)C1=COCCC1', 'O=C1Nc2ccccc2Cc2ccccc21', 'O=c1[nH]c(=O)n(-c2ccccc2)o1', 'O=c1oc2ccccc2cc1C1CCCc2ccccc21', 'O=C1NC(=O)C2(CCC2)C(=O)N1', 'O=C1NC(=O)C(C2=CCC3CCC2C3)C(=O)N1', 'c1ccc2nc(N3CCNCC3)ncc2c1', 'c1ccc2c(c1)CCC1C2CCC2CCCC21', 'O=C1C2C3C4CC5C3C1C5C42', 'O=C1CNC(=O)N1c1ccccc1', 'c1ncncn1', 'C1CO1', 'c1ccc2cc3c(cc2c1)-c1cccc2cccc-3c12', 'c1ccc(NNc2ccccc2)cc1', 'c1cc2ccc3ccc4ccc5cccc6c(c1)c2c3c4c56', 'C1=CC2CC1C1C3CC(C4OC34)C21', 'O=c1nccc[nH]1', 'O=C1CCCC

In [14]:
scaffold_list = []

for smiles in smiles_df["smiles"]:
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        scaffold_list.append(None)
        continue

    scaffold_mol = MurckoScaffold.GetScaffoldForMol(mol)
    scaffold_smiles = Chem.MolToSmiles(scaffold_mol)

    if scaffold_smiles == "":
        scaffold_list.append(smiles)
    else:
        scaffold_list.append(scaffold_smiles)

smiles_df["scaffold"] = scaffold_list



In [15]:
# smiles_df = smiles_df.drop(columns="scafold")
# print(smiles_df.head(5))

In [16]:
grouped = smiles_df.groupby("scaffold")

In [17]:
# each iteration over groupby object gives (scaffold key, group_df)

scaffold_indices_tuples = []

for scaffold, group_df in grouped:
    row_indices = group_df.index.to_list()

    scaffold_indices_tuples.append((scaffold, row_indices))


    # print(f"Key: {scaffold}\n")
    # print(f"Rows: {row_indices}\n")
    # print(f"Group df: {group_df}\n")

print(scaffold_indices_tuples[0])


('BrC(Br)(Br)Br', [722])


In [18]:
sorted_scaffold_indices_list = sorted(scaffold_indices_tuples,
                                       key= lambda item: len(item[1]), 
                                       reverse=True)

In [19]:
total_row_count = smiles_df.shape[0]

train_indices = []
test_indices = []

target_train_size = 0.80 * total_row_count

for scaffold, indices in sorted_scaffold_indices_list:
    group_size = len(indices)
    current_train_size = len(train_indices)

    if current_train_size + group_size <= target_train_size:
        train_indices.extend(indices)
    else:
        test_indices.extend(indices)


In [20]:
X = smiles_df[[#
    "mol_weight",
    "logp",
    "tpsa",
    "h_donors",
    "h_acceptors"
]]

y = smiles_df["y"]

x_train_sc = X.loc[train_indices]
x_test_sc = X.loc[test_indices]
y_train_sc = y.loc[train_indices]
y_test_sc = y.loc[test_indices]

In [21]:
model = RandomForestRegressor(n_estimators=400, # number of trees
                              random_state=42,
                              n_jobs=-1) # -1 to use all processors without jobs in parallel

In [22]:
model.fit(x_train_sc, y_train_sc) # trained the model

y_pred_sc = model.predict(x_test_sc)

results = pd.DataFrame({
    "Predicted": y_pred_sc,
    "True": y_test_sc.to_numpy()
})

print(results.head(5))

   Predicted   True
0   0.127045  1.110
1  -0.520363  0.940
2  -2.152166 -3.900
3  -3.592032  1.144
4  -2.091136  0.651


In [23]:
rmse_sc = mean_squared_error(y_pred=y_pred_sc, y_true=y_test_sc)**0.5
r2_sc = r2_score(y_pred=y_pred_sc, y_true=y_test_sc)

print(f"RMSE: {rmse_sc}\nR^2: {r2_sc}")

RMSE: 1.0268967589153313
R^2: 0.7961939767337751


In [24]:
print("x_train_sc shape:", x_train_sc.shape)
print("x_test_sc shape:", x_test_sc.shape)
print("index of x_train_sc matches train_indices:", list(x_train_sc.index) == train_indices)

x_train_sc shape: (902, 5)
x_test_sc shape: (226, 5)
index of x_train_sc matches train_indices: True


In [25]:
print("sum of train_indices:", sum(train_indices))
print("sorted first 10 train_indices:", sorted(train_indices)[:10])
print("sorted first 10 test_indices:", sorted(test_indices)[:10])

sum of train_indices: 505335
sorted first 10 train_indices: [2, 5, 6, 7, 8, 11, 12, 14, 15, 16]
sorted first 10 test_indices: [0, 1, 3, 4, 9, 10, 13, 21, 33, 36]


In [26]:
print(model.get_params())

print(smiles_df.loc[[2, 0, 5, 1], ["mol_weight", "logp", "tpsa", "h_donors", "h_acceptors", "y"]])

{'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': None, 'max_features': 1.0, 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 400, 'n_jobs': -1, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}
   mol_weight     logp    tpsa  h_donors  h_acceptors     y
2     152.237  2.87800   17.07         0            1 -2.06
0     457.432 -3.10802  202.32         7           12 -0.77
5     135.191  2.29630   12.89         0            2 -1.50
1     201.225  2.84032   42.24         1            2 -3.30


# Molecular Graph - comparison DNN to the ML QSAR baseline

In [27]:
import torch
from torch_geometric.data import Data

/home/professor/miniconda3/envs/dataenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [28]:
def smiles_to_graph(smiles) -> Data:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None: 
        return None

    # node features
    node_features = []
    num_atoms = 0
    for atom in mol.GetAtoms():
        feature_vector = [atom.GetAtomicNum(), #
                          atom.GetDegree(),
                          atom.GetFormalCharge(),
                          atom.GetIsAromatic(),
                          atom.GetTotalNumHs()]
        node_features.append(feature_vector) # row per atom
        num_atoms += 1

    # edges features
    edge_index_pairs = []
    edge_features = []

    # print([type(value) for value in node_features[0]])

    num_edges = 0
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bond_feature = [bond.GetBondTypeAsDouble(), bond.GetIsAromatic()]

        edge_index_pairs.append((i, j))
        edge_features.append(bond_feature)

        edge_index_pairs.append((j, i)) # graph is directed but the bond isn't so we double the indices
        edge_features.append(bond_feature)
        num_edges += 1

    node_tensor = torch.tensor(node_features, dtype=torch.float32).reshape(num_atoms, 5) # we have only 5 features per atom 
    #edge_index_pairs_tensor = torch.tensor(edge_index_pairs).reshape(2, num_edges * 2) # we have only 2 points per edge

############################################## if there was just a single element, then there are no bonds, so we need to trigger the special condition to make an empty 2d tensor
    #edge_index_pairs_tensor = torch.tensor(edge_index_pairs).transpose(0,1) 
    if edge_index_pairs:
        edge_index_pairs_tensor = torch.tensor(edge_index_pairs).transpose(0,1).contiguous()
        edge_tensor = torch.tensor(edge_features).reshape(num_edges*2, 2) # we have only 2 features per bond
    else:
        edge_index_pairs_tensor = torch.empty((2,0), dtype=torch.long)
        edge_tensor = torch.empty((0,2), dtype= torch.float32)

##############################################


    return Data(x=node_tensor,
                edge_index=edge_index_pairs_tensor,
                edge_attr=edge_tensor)


In [29]:
g = smiles_to_graph("c1ccccc1")
print(g.x.shape)
print(g.x.dtype)
print("\n")
print(g.edge_index)
print(g.edge_attr)


torch.Size([6, 5])
torch.float32


tensor([[0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 0],
        [1, 0, 2, 1, 3, 2, 4, 3, 5, 4, 0, 5]])
tensor([[1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000],
        [1.5000, 1.0000]])


In [30]:
graphs = []

for row in smiles_df.index:
    smiles = smiles_df.loc[row, "smiles"]
    target = smiles_df.loc[row, "y"]

    g = smiles_to_graph(smiles)
    g.y = torch.tensor([target], dtype=torch.float) # attach y into the Data object

    # print(row)
    # print('\n')
    # print(g)

    graphs.append(g) # pointer to the graph at the appropriate row index

train_graphs = [graphs[i] for i in train_indices]
test_graphs = [graphs[i] for i in test_indices]

print(len(train_graphs))
print('\n')
print(len(test_graphs))

902


226


In [31]:
# print(smiles_df["smiles"][934])
# print('\n')
# print(smiles_df["y"][934])

In [32]:
# g = smiles_to_graph("C")
# print(g.edge_index)
# print(g.edge_index.dtype)

# g = smiles_to_graph("CCO")
# print(g.edge_index)
# print(g.edge_index.dtype)

# Graph Neural Network Implementation

In [33]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool

In [34]:
# GNN Class implementation
# don't forget to implement global pooling at the end before giving it to regression

class MoleculeGNN(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        # initialize layers
        self.conv1 = GCNConv(in_dim, hidden_dim) # input mapped to output which is a hidden dim
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.output_head = nn.Linear(hidden_dim, 1) # a single dimention for output - just a scalar
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x, edge_index, batch): # here we need batch for pooling to tell wich atom belong to which molecule
        x = self.conv1(x, edge_index)
        x = F.relu(x) # after a layer, we need activation to make a decision and choose whether the signal is passing through this neuron or not
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        # now we collapse the learned whatever into a single feature vector from all the atoms in a molecule
        x = self.dropout(x)
        x = global_mean_pool(x, batch)
        prediction = self.output_head(x) # we already specified the output layer dimention in the init

        return prediction


In [35]:
# g = smiles_to_graph("c1ccccc1")
# model = MoleculeGNN(in_dim=5, hidden_dim=32)

# batch = torch.zeros(g.x.shape[0], dtype=torch.long) # for every atom, which molecule does it belong to

# output = model(g.x, g.edge_index, batch) # run the forward pass on the model

# print(output)
# print(output.shape)

# Training the GNN

In [36]:
from torch_geometric.loader import DataLoader as DL

In [37]:
dl = DL(train_graphs, batch_size=32, shuffle=True)

In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MoleculeGNN(in_dim=5, hidden_dim=16).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=1e-4)
loss_fn = nn.MSELoss()

In [39]:
# one epoch for now


num_epochs = 70


for epoch in range(num_epochs):

    model.train()
    num_batches = 0
    epochs_loss_total = 0

    for batch in dl:
        pred= model(batch.x, batch.edge_index, batch.batch)
        prediction = pred.squeeze(-1)
        #print("prediction shape:", prediction.shape, "| batch.y shape:", batch.y.shape)

        optimizer.zero_grad()  # clear old gradients if any
        loss = loss_fn(prediction, batch.y) # new gradient by backprop
        #print("loss:", (loss.item())**0.5)

        loss.backward()
        optimizer.step()  # update weights using those gradients
        epochs_loss_total += loss.item()
        num_batches += 1


    average_epoch_loss = epochs_loss_total / num_batches
    print(f"Number of batches: {num_batches}\n")
    print(f"epoch loss (MSE): {average_epoch_loss}, RMSE-equivalent: {average_epoch_loss ** 0.5}")


Number of batches: 29

epoch loss (MSE): 6.465019850895323, RMSE-equivalent: 2.5426403306199883
Number of batches: 29

epoch loss (MSE): 4.687153125631398, RMSE-equivalent: 2.1649834007750264
Number of batches: 29

epoch loss (MSE): 4.368234659063405, RMSE-equivalent: 2.0900322148386623
Number of batches: 29

epoch loss (MSE): 3.962643919319942, RMSE-equivalent: 1.9906390730918404
Number of batches: 29

epoch loss (MSE): 3.709592395815356, RMSE-equivalent: 1.9260302167451464
Number of batches: 29

epoch loss (MSE): 3.814192480054395, RMSE-equivalent: 1.9529957706186656
Number of batches: 29

epoch loss (MSE): 3.777718112386506, RMSE-equivalent: 1.9436352827592183
Number of batches: 29

epoch loss (MSE): 3.781792969539248, RMSE-equivalent: 1.9446832568671042
Number of batches: 29

epoch loss (MSE): 3.8486043009264717, RMSE-equivalent: 1.9617859977394252
Number of batches: 29

epoch loss (MSE): 3.6403829599248954, RMSE-equivalent: 1.9079787629648541
Number of batches: 29

epoch loss (MSE

# Testing the GNN

In [40]:
test_dl = DL(test_graphs, batch_size=32, shuffle=True)

In [41]:
model.eval()

all_predictions = []
all_targets = []

with torch.no_grad():
    for batch in test_dl:
        pred = model(batch.x, batch.edge_index, batch.batch).squeeze(-1)
        all_predictions.append(pred)
        all_targets.append(batch.y)

pred_tensor = torch.cat(all_predictions)
target_tensor = torch.cat(all_targets)
test_loss = loss_fn(pred_tensor, target_tensor)

print(f"MSE LOSS: {test_loss**0.5}")

MSE LOSS: 1.4208040237426758
